# Fixing a state

A chosen scissor mechanism is constructed, its length and hinge loations changed, 

Additional Dependency: Pygmo is currently not included in the quivira `environment.yml`.

In [ ]:
import numpy as np
import heyoka as hy
import pygmo as pg
import quivira as qv
from _animation import animate_bar_model
# Time measurement
import time

## 1. Build a scissor mechanism and a valid initial state

A 4-bar scissor mechansim is constructed. The bars are connected by joints, while two supports constrain the motion of the mechanism.

Joints only require the indices of the bars they join together and the respective position along the bars.
For a bar with midpoint position $(x, y)$ and angle $\theta$, a point on the bar is described by
$$
\mathbf{p}(s) = \begin{bmatrix} x \\ y \end{bmatrix}
+ s\,L \begin{bmatrix} \cos\theta \\ \sin\theta \end{bmatrix},
$$
where $L$ is the bar length and $s \in [-\tfrac{1}{2}, \tfrac{1}{2}]$ is the normalized position along the bar measured from its center. The endpoints correspond to $s = -\tfrac{1}{2}$ and $s = \tfrac{1}{2}$, while intermediate locations use any value in that interval.

The following variables are parametrized:

- par[0]-par[3]: The length of each bar
- par[4]-par[5]: The values $s$ of the first two bars that define the joint location
- par[6]: The force value driving the one degree of freedom of the mechanism

The ode system can already be built with the topology information alone.

We need an initial state, which is set in a convenient state of the mechanism, `state0`. It could be set in any other state as well.

In [ ]:
model = qv.bar_joint_model(tolerance=1e-8)
l0 = hy.par[0]
l1 = hy.par[1]
l2 = hy.par[2]
l3 = hy.par[3]
s_joint_bars_0_1 = hy.par[4]
s_joint_bars_1_0 = hy.par[5]
rho = 1

# Define the topology
# With lengths as heyoka parameters we can not use model.add_bar_from_point_pair() as it would infer a numerical value for length
model.add_bar(length=l0, mass=l0*rho)
model.add_bar(length=l1, mass=l1*rho)
model.add_bar(length=l2, mass=l2*rho)
model.add_bar(length=l3, mass=l3*rho)
# We can not use model.infer_joints(), as it requires numerical bar lengths for intersection detection
model.add_joint(bar_index=[0, 1], s=[s_joint_bars_0_1, s_joint_bars_1_0])
model.add_joint(bar_index=[0, 3], s=[0.5, -0.5])
model.add_joint(bar_index=[1, 2], s=[0.5, -0.5])
model.add_joint(bar_index=[2, 3], s=[0.5, 0.5])
# Add supports
model.add_support(bar_index=0, s=-0.5, dof_in_x=False, dof_in_y=False, x=0.0, y=0.0)
model.add_support(bar_index=1, s=-0.5, dof_in_x=False, dof_in_y=True, x=0.0, y=4.0)

# Add an external force at the beginning of bar 1
Fy = hy.par[6]
model.add_external_force(bar_index=1, s=-0.5, fx=0.0, fy=Fy)

# Build the ODE
ode_system = model.build_lagrange_eom()

# Define initial state
model.set_init_bar_attitude(bar_index=0, x0=2, y0=2, theta0=np.deg2rad(45))
model.set_init_bar_attitude(bar_index=1, x0=2, y0=2, theta0=np.deg2rad(-45))
model.set_init_bar_attitude(bar_index=2, x0=5, y0=1, theta0=np.deg2rad(45))
model.set_init_bar_attitude(bar_index=3, x0=5, y0=3, theta0=np.deg2rad(-45))
# Bar lengths:
param_values0 = [4 * np.sqrt(2), 4 * np.sqrt(2), 2 * np.sqrt(2), 2 * np.sqrt(2), 0.0, 0.0]
#Check the initial state
state0 = np.asarray(model.get_initial_state(), dtype=float)
model.check_constraint_fulfillment(state0, param_values0)
# Set the force value:
param_values0.append(50.0)

Basic collision events are defined to stop simulations early, to avoid singularities.

In [ ]:
# Terminal events: prevent joint/support coincidences
d_min = 0.5

# Symbolic positions of all joints, using their first connected bar.
# This automatically accommodates parametrized lengths and joint s values.
joint_positions = [
    model._expr_point_on_bar(j.bar_index[0], j.s[0])
    for j in model.joints
]

# Symbolic positions of the actual support points.
# Important: the slider's y coordinate is free to move.
support_positions = [
    model._expr_point_on_bar(s.bar_index, s.s)
    for s in model.supports
]

# Pairs to monitor
collision_pairs = [
    ("Joint 0 - Joint 3", joint_positions[0], joint_positions[3]),
    ("Joint 1 - Joint 2", joint_positions[1], joint_positions[2]),
    ("Joint 1 - Support 0", joint_positions[1], support_positions[0]),
    ("Joint 1 - Support 1", joint_positions[1], support_positions[1]),
    ("Joint 2 - Support 0", joint_positions[2], support_positions[0]),
    ("Joint 2 - Support 1", joint_positions[2], support_positions[1]),
]

collision_events = []

for name, (x1, y1), (x2, y2) in collision_pairs:
    if name == "Joint 2 - Support 0":
        # Only monitor absolute x separation
        distance_sq = (x1 - x2)**2
    else:
        # Monitor Euclidean distance
        distance_sq = (x1 - x2)**2 + (y1 - y2)**2

    collision_events.append(hy.t_event(distance_sq - d_min**2, direction=hy.event_direction.negative))

The taylor adaptive integrator can now be built.

With a positive force value in y, the state is advanced using the integrator to a "stowed" configuration called `state1`, stopped by the events defined above.

Reversing the force, setting the velocities of `state1` to 0, and integrating again leads to a deployed configuration `state2`.

The transition `state1` -> `state2` is animated.

In [ ]:
ta = hy.taylor_adaptive(
    ode_system,
    pars=param_values0,
    t_events=collision_events,
    compact_mode=True
)

ta.state[:] = state0
t_end = 20.0
tgrid = np.linspace(0.0, t_end, 401)
result1 = ta.propagate_grid(tgrid)

# Integrate until the first collision threshold is reached
solution1 = np.asarray(result1[-1], dtype=float)

# Save the exact final state and time of the first integration
state1 = ta.state.copy()
t1 = ta.time

# Stop the mechanism at the turning point
# This sets all generalized velocities to zero.
nq = len(model.q)
ta.state[nq:] = 0.0

# The new parameters are the old one but the force is reversed
param_values1 = np.asarray(param_values0, dtype=float).copy()
param_values1[6] = -50.0
ta.pars[:] = param_values1

# Integrate for another 20 seconds, starting at the current time
tgrid2 = t1 + np.linspace(0.0, t_end, 401)

result2 = ta.propagate_grid(tgrid2)
solution2 = np.asarray(result2[-1], dtype=float)

print("Final time:", ta.time)
print("Number of frames:", len(solution2))

animate_bar_model(
    model,
    solution2,
    pars=ta.pars,
    save_animation=False
)

## 2. Adjust the parameters

We adjust the coordinates s where the first hinge attaches to each of the bars, and the length of the other two bars.

In [ ]:
# Adjust heyoka parameters
param_values3 = np.asarray(ta.pars, dtype=float).copy()
param_values3[4] = +0.1
param_values3[5] = -0.1
param_values3[2] = param_values3[0]/2-0.1*param_values3[0]
param_values3[3] = param_values3[1]/2+0.1*param_values3[1]

## 3. Formulation as a constrained optimization problem

After changing the bar lengths and joint attachment positions, the previous configuration generally no longer satisfies the geometric constraints of the mechanism. We therefore need to find a new valid configuration that remains as close as possible to the original one.

We formulate this as a constrained optimization problem:

$$
\boxed{
\begin{aligned}
\min_{\mathbf q}\quad & D(\mathbf q,\mathbf q_{\mathrm{ref}})\
\text{subject to}\quad & \mathbf F(\mathbf q,\mathbf p_{\mathrm{new}})=0
\end{aligned}
}
$$

Here:

$\mathbf q_{\mathrm{ref}}$ is the configuration before reconfiguration.

$\mathbf q$ contains the new bar positions and orientations.

$\mathbf p_{\mathrm{new}}$ contains the updated geometric parameters, including bar lengths and joint attachment coordinates.

$\mathbf F=0$ enforces the geometric constraints of the mechanism.

$D$ measures how much the configuration changes.

The objective is to minimize the movement of the mechanism while satisfying all geometric constraints.

Furthermore, we define the Jacobian $\mathbf J=\partial\mathbf F/\partial\mathbf q$, which will tell the optimizer how the constraints change when the bar coordinates change.

In [ ]:
# Reference configuration and adjustments
nq = len(model.q)
# Position from the stowed state
q_ref = np.asarray(state1[:nq], dtype=float).copy()

# Constraint expressions and number of equality constraints
F_sym = list(model.F)
m = len(F_sym) # FOr reshaping J_num and to tell Pygmo how many eqauality constraints we have

# Jacobian: derivatives of each constraint with respect to all coordinates
J_sym = []
for f in F_sym:
    for q in model.q:
        J_sym.append(hy.diff(f, q))

# Compile only once; these functions are reused by both objectives
F_func = hy.cfunc(F_sym, vars=model.q, compact_mode=True)
J_func = hy.cfunc(J_sym, vars=model.q, compact_mode=True)

# Only parameters 0 through 5 occur in the geometric constraints
geometry_pars = np.asarray(param_values3[:6], dtype=float)

def F_num(q):
    return np.asarray(F_func(q, pars=geometry_pars), dtype=float)

def J_num(q):
    values = np.asarray(J_func(q, pars=geometry_pars), dtype=float)
    return values.reshape(m, nq)

We define a common StateProjection class as expeced by Pygmo to compare different distance metrics and optimization algorithms.

The class accepts an objective function and its gradient, allowing us to switch between different metrics while keeping the geometric constraints, Jacobian, and search bounds unchanged.

This also allows us to test different Pygmo solvers on the same problem and compare their accuracy, computational cost, and robustness.

In [ ]:
class StateProjection:
    def __init__(self, objective, objective_gradient):
        self.objective = objective
        self.objective_gradient = objective_gradient

    def fitness(self, q):
        # Pygmo expects [objective, equality_constraint_0, ...]
        objective_value = self.objective(q)
        constraints = F_num(q)
        return np.concatenate(([objective_value], constraints))

    def get_nec(self):
        return m

    def get_bounds(self):
        # Keep the same bounds for all approaches
        width = 2.0 * max(param_values3[:4])
        return ((q_ref - width).tolist(), (q_ref + width).tolist())

    def gradient(self, q):
        # First the objective gradient; then the flattened constraint Jacobian
        grad_obj = self.objective_gradient(q)
        J = J_num(q)
        return np.concatenate((grad_obj, J.ravel()))

In [ ]:
solver_names = [
    "NLopt SLSQP",
    "SciPy SLSQP",
    "SciPy trust-constr",
    "NLopt COBYLA",
    "IPOPT",
]

def make_solver(name):
    # Create a new solver for each independent run
    if name == "NLopt SLSQP":
        solver = pg.nlopt("slsqp")
        solver.xtol_rel = 1e-11
        solver.maxeval = 1000
    elif name == "SciPy SLSQP":
        solver = pg.scipy_optimize(
            method="SLSQP", options={"ftol": 1e-11, "maxiter": 1000}
        )
    elif name == "SciPy trust-constr":
        solver = pg.scipy_optimize(
            method="trust-constr", options={"gtol": 1e-10, "maxiter": 1000}
        )
    elif name == "NLopt COBYLA":
        solver = pg.nlopt("cobyla")
        solver.xtol_rel = 1e-10
        solver.maxeval = 5000
    elif name == "IPOPT":
        solver = pg.ipopt()  # Available only in some Pygmo builds
        solver.set_numeric_option("tol", 1e-10)
        solver.set_integer_option("max_iter", 1000)
    else:
        raise ValueError(f"Unknown solver: {name}")
    return solver


def benchmark_projection(label, objective, objective_gradient):
    results = []

    for name in solver_names:
        row = {"approach": label, "solver": name, "feasible": False}
        try:
            problem = pg.problem(StateProjection(objective, objective_gradient))
            problem.c_tol = [min(1e-9, model.tolerance)] * m

            # A fresh one-individual population, always starting at q_ref
            pop = pg.population(problem, size=0)
            pop.push_back(q_ref.tolist())
            fevals_before = pop.problem.get_fevals()
            gevals_before = pop.problem.get_gevals()

            solver = make_solver(name)
            start = time.perf_counter()
            pop = pg.algorithm(solver).evolve(pop)
            elapsed = time.perf_counter() - start

            q_result = np.asarray(pop.get_x()[0], dtype=float).copy()
            residual = float(np.max(np.abs(F_num(q_result))))
            value = float(objective(q_result))

            row.update({
                "q": q_result,
                "objective": value,
                "residual": residual,
                "feasible": bool(np.isfinite(residual) and
                                 residual <= model.tolerance),
                "time_s": elapsed,
                "fevals": pop.problem.get_fevals() - fevals_before,
                "gevals": pop.problem.get_gevals() - gevals_before,
            })
        except Exception as exc:
            # For example: IPOPT is not included in the local installation
            row["error"] = str(exc)

        results.append(row)

    print(f"\n{label}")
    print(f"{'Solver':<21} {'Feasible':<9} {'Objective':>12} "
          f"{'Max |F|':>12} {'Time [s]':>10} {'F evals':>9}")
    for row in results:
        if "error" in row:
            print(f"{row['solver']:<21} unavailable/failed: {row['error']}")
        else:
            print(f"{row['solver']:<21} {str(row['feasible']):<9} "
                  f"{row['objective']:>12.6g} {row['residual']:>12.3e} "
                  f"{row['time_s']:>10.4f} {row['fevals']:>9}")

    return results


def best_feasible(results):
    valid = [row for row in results if row["feasible"]]
    if not valid:
        raise RuntimeError("No solver found a configuration within model.tolerance")
    return min(valid, key=lambda row: row["objective"])

Helper functions:

In [ ]:
def build_state3(q_new):
    state3 = np.concatenate((np.asarray(q_new, dtype=float), np.zeros(nq)))
    model.check_constraint_fulfillment(state3, pars=geometry_pars)
    return state3


def simulate_state3(state3):
    # Restart from the time at which state1 was saved
    ta.pars[:] = param_values3
    ta.state[:] = state3
    ta.time = t1
    ta.reset_cooldowns()

    tgrid3 = t1 + np.linspace(0.0, t_end, 401)
    result3 = ta.propagate_grid(tgrid3)
    solution3 = np.asarray(result3[-1], dtype=float)

    print("Integration outcome:", result3[0])
    print("Number of frames:", len(solution3))

    if len(solution3) > 0:
        animate_bar_model(model, solution3, pars=ta.pars, save_animation=False)

    return solution3

Since generalized coordinates contain both translations (meters) and rotations (radians), a simple Euclidean distance between coordinate vectors is not physically meaningful. We therefore consider two geometrically motivated distance measures based on the displacement of bar endpoints: a quadratic approximation and an exact endpoint-displacement metric.

## 4. Defining the optimization metrics from the movement of the endpoints

The endpoint positions are

$$
\boxed{
\mathbf r^\pm(\mathbf q,L)
=
\begin{bmatrix}x\\y\end{bmatrix}
\pm\frac L2
\begin{bmatrix}\cos\theta\\\sin\theta\end{bmatrix}
}
$$

For convenience, define the orientation vector

$$
\mathbf u(\theta)=
\begin{bmatrix}\cos\theta\\\sin\theta\end{bmatrix}.
$$

Then

$$
\mathbf r^\pm=\mathbf c\pm\frac L2\mathbf u(\theta).
$$

Now we calculate the displacement between the two configurations

Denoting the state before adjusting the parameters as 0, and the new state as 1, we define

$$
\Delta\mathbf c=\mathbf c_1-\mathbf c_0,\qquad
\Delta\theta=\theta_1-\theta_0.
$$

The displacement of each endpoint is

$$
\Delta\mathbf r^\pm
=
\mathbf r^\pm_{\mathrm{new}}-\mathbf r^\pm_{\mathrm{ref}}.
$$

Substituting the endpoint equations:

$$
\boxed{
\Delta\mathbf r^\pm
=
\Delta\mathbf c
\pm\frac12
\left[
L_1\mathbf u(\theta_1)
-
L_0\mathbf u(\theta_0)
\right].
}
$$

Now we can define the distance metric. We choose the average squared displacement of the two endpoints:

$$
\boxed{
D=
\frac12\left(
\|\Delta\mathbf r^-\|^2+
\|\Delta\mathbf r^+\|^2
\right).
}
$$

Introduce the vector

$$
\mathbf a=
\frac12
\left[
L_1\mathbf u(\theta_1)-
L_0\mathbf u(\theta_0)
\right].
$$

Then

$$
\Delta\mathbf r^-=\Delta\mathbf c-\mathbf a,
\qquad
\Delta\mathbf r^+=\Delta\mathbf c+\mathbf a.
$$

Substituting into the objective:

$$
D=
\frac12\left[
\|\Delta\mathbf c-\mathbf a\|^2+
\|\Delta\mathbf c+\mathbf a\|^2
\right].
$$

Expanding:

$$
\begin{aligned}
D=\frac12\big[&
\|\Delta\mathbf c\|^2-2\Delta\mathbf c\cdot\mathbf a+\|\mathbf a\|^2\\
+&\|\Delta\mathbf c\|^2+2\Delta\mathbf c\cdot\mathbf a+\|\mathbf a\|^2
\big].
\end{aligned}
$$

The cross terms cancel:

$$
\boxed{D=\|\Delta\mathbf c\|^2+\|\mathbf a\|^2.}
$$

Inserting $\mathbf a$, we obtain

$$
\boxed{
D=
\Delta x^2+\Delta y^2+
\frac14
\left\|
L_1\mathbf u(\theta_1)-
L_0\mathbf u(\theta_0)
\right\|^2.
}
$$

### 4.1 Exact endpoint metric

We now evaluate the remaining squared norm:

$$
\left\|
L_1\mathbf u(\theta_1)-
L_0\mathbf u(\theta_0)
\right\|^2.
$$

Expanding the scalar product:

$$
\begin{aligned}
&\left(
L_1\mathbf u(\theta_1)-L_0\mathbf u(\theta_0)
\right)^T
\left(
L_1\mathbf u(\theta_1)-L_0\mathbf u(\theta_0)
\right)\\
&=L_1^2\|\mathbf u(\theta_1)\|^2+
L_0^2\|\mathbf u(\theta_0)\|^2
-2L_1L_0\mathbf u(\theta_1)^T\mathbf u(\theta_0).
\end{aligned}
$$

Because the orientation vectors have unit length,

$$
\|\mathbf u(\theta_1)\|^2
=
\|\mathbf u(\theta_0)\|^2=1.
$$

The dot product of the two orientation vectors is

$$
\begin{aligned}
\mathbf u(\theta_1)^T\mathbf u(\theta_0)
&=\cos\theta_1\cos\theta_0+
\sin\theta_1\sin\theta_0\\
&=\cos(\theta_1-\theta_0)\\
&=\cos\Delta\theta.
\end{aligned}
$$

Therefore,

$$
\left\|
L_1\mathbf u(\theta_1)-L_0\mathbf u(\theta_0)
\right\|^2
=
L_1^2+L_0^2-2L_1L_0\cos\Delta\theta.
$$

We obtain the **exact endpoint metric**:

$$
\boxed{
D_{\mathrm{end}}=
\Delta x^2+\Delta y^2+
\frac{L_1^2+L_0^2-2L_1L_0\cos\Delta\theta}{4}.
}
$$

This expression measures the actual squared distance between the old and new endpoint positions, even when the bar rotates substantially or changes length. Its derivative is:

$$
\boxed{
\nabla_{\mathbf q}D_{\mathrm{end}}
=
\begin{bmatrix}
2\Delta x\\
2\Delta y\\
\frac{L_1L_0}{2}\sin\Delta\theta
\end{bmatrix}.
}
$$

In [ ]:
# state1 was obtained with the original bar lengths (param_values0)
L_old = np.asarray(param_values0[:4], dtype=float)
L_new = np.asarray(param_values3[:4], dtype=float)


def objective_endpoints(q):
    dq = np.asarray(q, dtype=float) - q_ref

    # Centers: squared translations of all bars
    translation = np.sum(dq[0::3]**2 + dq[1::3]**2)

    # Endpoints: exact contribution from length and orientation changes
    dtheta = dq[2::3]
    endpoint_rotation = (L_new**2 + L_old**2 - 2.0 * L_new * L_old * np.cos(dtheta)) / 4.0

    return float(translation + np.sum(endpoint_rotation))


def gradient_endpoints(q):
    dq = np.asarray(q, dtype=float) - q_ref
    grad = np.zeros(nq)

    grad[0::3] = 2.0 * dq[0::3]
    grad[1::3] = 2.0 * dq[1::3]
    grad[2::3] = 0.5 * L_new * L_old * np.sin(dq[2::3])

    return grad

We now solve the state-projection problem using the exact endpoint metric as our objective. We run all selected optimization algorithms and choose the **feasible solution with the lowest objective value**.

In [ ]:
results_end = benchmark_projection(
    "Exact endpoints", objective_endpoints, gradient_endpoints
)
best_end = best_feasible(results_end)

q_new_end = best_end["q"].copy()
state3_end = build_state3(q_new_end)

print("Selected solver:", best_end["solver"])
print("Position correction:", np.linalg.norm(q_new_end - q_ref))
print("Maximum constraint violation:", best_end["residual"])

# Simulate and animate this version of state3
solution3_end = simulate_state3(state3_end)

### 4.2 Approximate Endpoint Metric

Return to the **exact endpoint metric**:

$$
D_{\mathrm{end}}=
\Delta x^2+\Delta y^2+
\frac{L_1^2+L_0^2-2L_1L_0\cos\Delta\theta}{4}.
$$

Now apply the first order taylor expansion small-angle approximation to the cosine:

$$
\cos\Delta\theta\approx1-\frac{\Delta\theta^2}{2}.
$$

We obtain

$$
\begin{aligned}
D_{\mathrm{end}}
&\approx\Delta x^2+\Delta y^2\\
&\quad+\frac14\left[
L_1^2+L_0^2-
2L_1L_0\left(1-\frac{\Delta\theta^2}{2}\right)
\right].
\end{aligned}
$$

Expanding:

$$
D_{\mathrm{end}}
\approx
\Delta x^2+\Delta y^2+
\frac{L_1^2+L_0^2-2L_1L_0}{4}
+
\frac{L_1L_0}{4}\Delta\theta^2.
$$

Since

$$
L_1^2+L_0^2-2L_1L_0=(L_1-L_0)^2,
$$

the result becomes

$$
D_{\mathrm{end}}
\approx
\Delta x^2+\Delta y^2+
\frac{(L_1-L_0)^2}{4}+
\frac{L_1L_0}{4}\Delta\theta^2.
$$

Since $(L_1-L_0)^2/2$ is constant, it can be omitted from the optimization:

$$
\boxed{
D_{\mathrm{end}}
\approx
\Delta x^2+\Delta y^2+
\frac{L_1L_0}{4}\Delta\theta^2.
}
$$

We now have a quadratic approximation of the endpoint displacement. Although we could use this expression directly as our optimization objective, it is convenient to represent it as a **quadratic form**:

$$
D_R = \frac12 \Delta\mathbf q^T \mathbf R \Delta\mathbf q,
$$

where

$$
\Delta\mathbf q =
\mathbf q-\mathbf q_{\mathrm{ref}}
=
\begin{bmatrix}
\Delta x\\
\Delta y\\
\Delta\theta
\end{bmatrix}.
$$

The matrix $\mathbf R$ weights the changes in generalized coordinates, accounting for the fact that translations and rotations have different physical effects. In particular, the rotational weight depends on both the old and new bar lengths.

Comparing the quadratic form

$$
D_R=\frac12\left(
R_{11}\Delta x^2+
R_{22}\Delta y^2+
R_{33}\Delta\theta^2
\right)
$$

with our derived objective

$$
D_R=\Delta x^2+\Delta y^2+
\frac{L_1L_0}{4}\Delta\theta^2,
$$

we obtain

$$
\boxed{
\mathbf R=\operatorname{diag}\left(
2,\;2,\;\frac{L_1L_0}{2}
\right).
}
$$

The factors of 2 compensate for the prefactor $1/2$ in the quadratic form.

**Why use this representation?** Since $\mathbf R$ is symmetric and constant during optimization, the objective gradient becomes particularly simple:

$$
\boxed{
\nabla_{\mathbf q}D_R
=
\mathbf R(\mathbf q-\mathbf q_{\mathrm{ref}}).
}
$$

For the complete mechanism, we construct a block-diagonal matrix containing one such $\mathbf R_i$ for each bar. Because all blocks are diagonal, we only need to store the diagonal entries as `R_diag`.

This gives us a computationally inexpensive objective and analytical gradient for Pygmo. Unlike the exact endpoint metric, the quadratic approximation does not require evaluating trigonometric functions during optimization.


In [ ]:
# Same coordinate ordering as in the model: x, y, theta for each bar
R_diag = []

for l_old, l_new in zip(L_old, L_new):
    R_diag.extend([2.0, 2.0, l_old * l_new / 2.0])

R_diag = np.asarray(R_diag, dtype=float)

def objective_R(q):
    dq = np.asarray(q, dtype=float) - q_ref
    return 0.5 * np.dot(R_diag * dq, dq)


def gradient_R(q):
    dq = np.asarray(q, dtype=float) - q_ref
    return R_diag * dq

We now solve the state-projection problem using the quadratic $R$-metric as our objective. We run all selected optimization algorithms and choose the **feasible solution with the lowest objective value**.

In [ ]:
results_R = benchmark_projection("Quadratic R-metric", objective_R, gradient_R)
best_R = best_feasible(results_R)

q_new_R = best_R["q"].copy()
state3_R = build_state3(q_new_R)

print("Selected solver:", best_R["solver"])
print("Position correction:", np.linalg.norm(q_new_R - q_ref))
print("Maximum constraint violation:", best_R["residual"])

# Simulate and animate this version of state3
solution3_R = simulate_state3(state3_R)

### 4.3 Comparison of the approaches

Note that the difference between the Quadratic cost and exact endpoint costs stems from the omitted constant term adding: 
$$
\sum_{i=0}^{3}\frac{(L_{1,i}-L_{0,i})^2}{4}=0.16.
$$

In [ ]:
print(f"{'Optimized metric':<20} {'Solver':<21} {'Quadr cost':>12} "
      f"{'Ex endp cost':>12} {'Max |F|':>12} {'Time [s]':>10}")

for row in results_end + results_R:
    if not row["feasible"]:
        continue
    q = row["q"]
    print(f"{row['approach']:<20} {row['solver']:<21} "
          f"{objective_R(q):>12.6g} {objective_endpoints(q):>12.6g} "
          f"{row['residual']:>12.3e} {row['time_s']:>10.4f}")

print("\nDifference between selected configurations:",
      np.linalg.norm(q_new_R - q_new_end))

In [ ]:

# 5. Find a valid configuration from a randomized initial condition

rng = np.random.default_rng(42)

# # Generate a random configuration near the reference configuration
# q_random = q_ref.copy()

# # Randomly perturb bar center positions and orientations
# q_random[0::3] += rng.normal(0.0, 0.5, size=len(model.bars))
# q_random[1::3] += rng.normal(0.0, 0.5, size=len(model.bars))
# q_random[2::3] += rng.normal(0.0, 0.2, size=len(model.bars))

n_bars = len(model.bars)
L_scale = max(L_new)

# Generate independent random positions and orientations
q_random = np.zeros(nq)

q_random[0::3] = rng.uniform(-L_scale, 2 * L_scale, n_bars)
q_random[1::3] = rng.uniform(-L_scale, 2 * L_scale, n_bars)
q_random[2::3] = rng.uniform(-np.pi, np.pi, n_bars)

print("Initial constraint violation:",
      np.max(np.abs(F_num(q_random))))


# Minimize the correction relative to the random configuration
# using the quadratic R-metric
def objective_random(q):
    dq = np.asarray(q, dtype=float) - q_random
    return 0.5 * np.dot(R_diag * dq, dq)


def gradient_random(q):
    dq = np.asarray(q, dtype=float) - q_random
    return R_diag * dq


# Create the constrained optimization problem
problem = pg.problem(
    StateProjection(objective_random, gradient_random)
)
problem.c_tol = [model.tolerance] * m

# Start the optimizer at the invalid random configuration
pop = pg.population(problem, size=0)
pop.push_back(q_random.tolist())

# Solve using SLSQP
solver = pg.nlopt("slsqp")
solver.xtol_rel = 1e-11
solver.maxeval = 2000

pop = pg.algorithm(solver).evolve(pop)

# Extract the corrected configuration
q_valid = np.asarray(pop.get_x()[0], dtype=float).copy()

print("Final constraint violation:",
      np.max(np.abs(F_num(q_valid))))

print("Coordinate correction:",
      np.linalg.norm(q_valid - q_random))

# Construct a complete state with zero velocities
state_valid = build_state3(q_valid)

# Simulate and animate the valid configuration
solution_valid = simulate_state3(state_valid)
